In [1]:
import openmc
import os

os.chdir('/home/eric/Documents/openMC_models/HTR-10-Pebble-Bed')
openmc.config['cross_sections'] = '/home/eric/openmc/nuclear_data/endf-b8.0-hdf5/endfb-viii.0-hdf5/cross_sections.xml'

In [2]:
with openmc.StatePoint('statepoint.200.h5') as sp:
    tally = sp.get_tally(name='rz_flux_map')
    mesh = sp.meshes[3]
    
    # Get thermal flux (first energy group, below 0.625 eV)
    flux = tally.get_reshaped_data(expand_dims=True).squeeze()
    print('flux shape:', flux.shape)
    print('flux min:', flux.min())
    print('flux max:', flux.max())

flux shape: (25, 40, 2)
flux min: 0.00026840660487250945
flux max: 0.8370583013538494


In [3]:
import pyvista as pv
pv.set_jupyter_backend('trame')
pv.global_theme.trame.server_proxy_enabled = True
pv.global_theme.trame.server_proxy_prefix = '/proxy/'

thermal_flux = flux[:, :, 0].reshape(25, 1, 40)
print('thermal flux shape:', thermal_flux.shape)
print('thermal flux min:', thermal_flux.min())
print('thermal flux max:', thermal_flux.max())


thermal flux shape: (25, 1, 40)
thermal flux min: 0.021300830730083535
thermal flux max: 0.5843007460902447


In [4]:
import numpy as np

# Create a new cylindrical mesh with the same r and z but more phi segments
n_phi = 24  # 24 segments gives a smooth-looking cylinder
vis_mesh = openmc.CylindricalMesh(
    r_grid=mesh.r_grid,
    z_grid=mesh.z_grid,
    phi_grid=np.linspace(0, 2*np.pi, n_phi + 1)
)

# Tile the thermal flux data across all phi segments
thermal_flux_3d = flux[:, :, 0].reshape(25, 1, 40)
tiled_flux = np.tile(thermal_flux_3d, (1, n_phi, 1))
print('tiled flux shape:', tiled_flux.shape)

tiled flux shape: (25, 24, 40)


In [5]:
vis_mesh.plot(datasets={'thermal_flux': tiled_flux},
              volume_normalization=False,
              threshold=0.001,
              log_scale=True,
              title='HTR-10 Thermal Flux Distribution',
              scalar_bar_title='Thermal Flux [n/cm²/s]')

/home/eric/openmc-fork/openmc/mesh.py:2335: UserWarning: Cartesian coordinates are returned from this property as of version 0.14.0
  warnings.warn('Cartesian coordinates are returned from this property as of version 0.14.0')


Widget(value='<iframe src="/proxy/37841/index.html?ui=P_0x7ba2a9e0b4a0_0&reconnect=auto" class="pyvista" style…